**שלב 1: טעינה חכמה ושמירה על המזהים המקוריים (IDs)**


בשלב זה נטען את הנתונים, נוודא שהתאריכים תקינים, ונוודא שאנחנו שומרים את ה-_id של כל שורה (גם של שאלון PANAS וגם של דיווח ה-EMA).

In [20]:
from pathlib import Path
import pandas as pd
import numpy as np
from pathlib import Path

current_dir = Path.cwd()
PROJECT_ROOT = current_dir.parent if current_dir.name == "notebooks" else current_dir
DATA_DIR = PROJECT_ROOT / "data" / "processed_parquet"

# בדיקה שהתיקייה אכן קיימת במחשב של המשתמשת
if not DATA_DIR.exists():
    raise FileNotFoundError(f"⚠️ התיקייה לא נמצאה בנתיב: {DATA_DIR}\nאנא ודאי שמבנה התיקיות בפרויקט תואם.")
else:
    print(f"✅ נתיב הנתונים נקלט בהצלחה: {DATA_DIR}")

✅ נתיב הנתונים נקלט בהצלחה: c:\Users\moria\Documents\projects\lifesnaps_project\data\processed_parquet


In [21]:
panas_raw = pd.read_parquet(DATA_DIR / "panas.parquet")
sema_raw = pd.read_parquet(DATA_DIR / "all_sema_emotions.parquet")

# 1.A. הכנת טבלת הבסיס ל-PANAS
panas_base = panas_raw.copy()
# יצירת המזהה המקורי לשאלון - נמנע משגיאת חוסר בעמודה (KeyError)
if '_id' in panas_base.columns:
    panas_base['panas_original_id'] = panas_base['_id'].astype(str)
elif 'id' in panas_base.columns and panas_base['id'].nunique() == len(panas_base): 
    panas_base['panas_original_id'] = panas_base['id'].astype(str)
else:
    panas_base['panas_original_id'] = ["panas_" + str(i) for i in range(len(panas_base))]

panas_user_col = next((col for col in ['user_id', 'participant_id', 'id'] if col in panas_base.columns), None)
panas_base['user_id'] = panas_base[panas_user_col].astype(str).str.strip()
panas_date_col = 'submitdate' if 'submitdate' in panas_base.columns else 'date'
panas_base['panas_date'] = pd.to_datetime(panas_base[panas_date_col], errors='coerce').dt.normalize()

# סינון שאלונים ריקים וחישוב פער ימים (ישמש את שתי הגישות)
panas_base = panas_base.dropna(subset=['panas_original_id', 'user_id', 'panas_date', 'positive_affect_score', 'negative_affect_score']).copy()
panas_base = panas_base.sort_values(by=['user_id', 'panas_date']).reset_index(drop=True)
panas_base['days_since_last_panas'] = panas_base.groupby('user_id')['panas_date'].diff().dt.days


# 1.B. הכנת טבלת הבסיס ל-EMA
ema_base = sema_raw.copy()
if '_id' in ema_base.columns:
    ema_base['ema_original_id'] = ema_base['_id'].astype(str)
elif 'id' in ema_base.columns and ema_base['id'].nunique() == len(ema_base):
    ema_base['ema_original_id'] = ema_base['id'].astype(str)
else:
    ema_base['ema_original_id'] = ["ema_" + str(i) for i in range(len(ema_base))]

ema_user_col = next((col for col in ['participant_id', 'user_id', 'data.PARTICIPANT_ID'] if col in ema_base.columns), None)
ema_base['user_id'] = ema_base[ema_user_col].astype(str).str.strip()
sema_ts_col = 'data.COMPLETED_TS' if 'data.COMPLETED_TS' in ema_base.columns else 'timestamp'
ema_base['ema_timestamp'] = pd.to_datetime(ema_base[sema_ts_col], errors='coerce').dt.tz_localize(None)
ema_base['ema_date'] = ema_base['ema_timestamp'].dt.normalize()

# סינון דיווחים ריקים או ללא רגש תקין
ema_base = ema_base.dropna(subset=['ema_original_id', 'user_id', 'ema_date', 'data.MOOD']).copy()
ema_base = ema_base[ema_base['data.MOOD'] != '<no-response>']

print(f"✅ שלב 1 הושלם. נטענו {len(panas_base)} שאלוני PANAS ו-{len(ema_base)} דיווחי EMA.\n")



✅ שלב 1 הושלם. נטענו 268 שאלוני PANAS ו-5036 דיווחי EMA.



In [ ]:
** שלב 2: גישה א' - מסגרת זמן קבועה ונוקשה (Rigid 7-Days Window) **
#
#בגישה זו אנו הולכים תמיד 7 ימים אחורה. לאחר מכן נסנן את האלונים
# שיש בהם חפיפה או שחסרים בהם נתונים מספקים (נצפה ל-165 שאלונים נקיים בסוף).

טיפול בבעיות שהתגלו ב-Sanity Checks
ביקשת לחשוב מה עושים עם החפיפות.
המלצה שלי למודל אמין: אם לנבדק יש שני שאלוני PANAS בהפרש של 3 ימים (לדוגמה ראשון ורביעי), הנתונים של ראשון-רביעי ייכנסו לשני המודלים וייצרו כפילות (Data Leakage פנימי). עדיף לסנן שאלונים חופפים ולהשאיר רק חלונות נקיים של 7 ימים. כמו כן, נסנן גם שבועות שיש בהם מעט מדי דיווחים (למשל פחות מ-3 דיווחים בשבוע, כי אי אפשר לייצג שבוע מדיווח בודד).

In [29]:
# =====================================================================
# שלב 2: גישה א' - מסגרת זמן קבועה ונוקשה (Rigid 7-Days Window)
# =====================================================================
# בגישה זו אנו הולכים תמיד 7 ימים אחורה. לאחר מכן נסנן את השאלונים
# שיש בהם חפיפה או שחסרים בהם נתונים מספקים (נצפה ל-165 שאלונים נקיים בסוף).
print("=== שלב 2: גישה א' - חלון זמן קבוע של 7 ימים (Rigid Window) ===")

rigid_records = []

for _, p_row in panas_base.iterrows():
    is_overlapping = pd.notna(p_row['days_since_last_panas']) and p_row['days_since_last_panas'] < 7
    start_date = p_row['panas_date'] - pd.Timedelta(days=7) # תמיד 7 ימים אחורה
    
    user_emas = ema_base[ema_base['user_id'] == p_row['user_id']]
    valid_emas = user_emas[(user_emas['ema_date'] > start_date) & (user_emas['ema_date'] <= p_row['panas_date'])]
    
    if len(valid_emas) > 0: # שומרים רק אם יש לפחות דיווח אחד ב-7 הימים
        for _, ema_row in valid_emas.iterrows():
            rigid_records.append({
                'user_id': p_row['user_id'],
                'panas_original_id': p_row['panas_original_id'],
                'panas_date': p_row['panas_date'],
                'positive_affect_score': p_row['positive_affect_score'],
                'negative_affect_score': p_row['negative_affect_score'],
                'is_overlapping_window': is_overlapping,
                'total_emas_in_window': len(valid_emas),
                'ema_original_id': ema_row['ema_original_id'],
                'data.MOOD': ema_row['data.MOOD']
            })

rigid_long_df = pd.DataFrame(rigid_records)


# ---------------------------------------------------------------------
# הרצת בדיקת שפיות ומעקב סינון נתונים (Data Funnel) עבור הגישה הקשיחה
# משתנים עם קידומת rigid_ למניעת דריסה עתידית
# ---------------------------------------------------------------------
print("\n=== מעקב סינון נתונים בגישה הקשיחה (Data Funnel) ===")

rigid_initial_count = len(panas_base)
print(f"1. התחלנו עם: {rigid_initial_count} שאלוני PANAS תקינים בבסיס.")

# 1. כמה שאלונים נמחקו כי לא היו להם דיווחי EMA בכלל?
rigid_with_any_ema = rigid_long_df['panas_original_id'].nunique()
rigid_no_ema_count = rigid_initial_count - rigid_with_any_ema
print(f"   -> נמחקו {rigid_no_ema_count} שאלונים כי לא היו להם דיווחי EMA בכלל ב-7 הימים שקדמו להם.")

# 2. מתוך אלו שנשארו, כמה נמחקו בגלל חפיפה עם שאלון קודם?
rigid_overlapping_count = rigid_long_df[rigid_long_df['is_overlapping_window'] == True]['panas_original_id'].nunique()
print(f"   -> נמחקו {rigid_overlapping_count} שאלונים בגלל חפיפת זמנים (פחות מ-7 ימים מהשאלון הקודם של הנבדק).")

# 3. מתוך אלו שלא חופפים, כמה נמחקו כי היו להם פחות מ-3 דיווחים?
rigid_sparse_count = rigid_long_df[(rigid_long_df['is_overlapping_window'] == False) & (rigid_long_df['total_emas_in_window'] < 3)]['panas_original_id'].nunique()
print(f"   -> נמחקו {rigid_sparse_count} שאלונים כי היו להם פחות מ-3 דיווחים (לא מספיק כדי לחשב ממוצע שבועי אמין).")

# חישוב הסך הכל המוזהב
rigid_final_gold_count = rigid_with_any_ema - rigid_overlapping_count - rigid_sparse_count
print(f"\n✅ נשארנו עם: {rigid_final_gold_count} שאלוני PANAS מוזהבים למודל בגישה הקשיחה.\n")


# ---------------------------------------------------------------------
# החלת מסנני האיכות (Sanity Checks) ויצירת הטבלה הארוכה הנקייה
# ---------------------------------------------------------------------
rigid_clean_long_df = rigid_long_df[
    (rigid_long_df['is_overlapping_window'] == False) & # ללא חפיפות
    (rigid_long_df['total_emas_in_window'] >= 3)        # לפחות 3 דיווחים לייצוג השבוע
].copy()


# ---------------------------------------------------------------------
# אגרגציה מפורמט ארוך לפורמט רחב (Wide Format) למודל
# ---------------------------------------------------------------------
mood_dummies_rigid = pd.get_dummies(rigid_clean_long_df['data.MOOD'], prefix='mood')
rigid_wide_prep = pd.concat([rigid_clean_long_df, mood_dummies_rigid], axis=1)

agg_funcs_rigid = {
    'user_id': 'first', 
    'panas_date': 'first', 
    'positive_affect_score': 'first', 
    'negative_affect_score': 'first', 
    'total_emas_in_window': 'first'
}
for col in mood_dummies_rigid.columns:
    agg_funcs_rigid[col] = 'mean' # חישוב התפלגות הרגשות (אחוז מתוך הדיווחים)

rigid_wide_df = rigid_wide_prep.groupby('panas_original_id').agg(agg_funcs_rigid).reset_index()

print(f"✅ אגרגציה הושלמה: נוצרה טבלת rigid_wide_df עם {len(rigid_wide_df)} תצפיות ייחודיות.\n")
display(rigid_wide_df.head(100))

=== שלב 2: גישה א' - חלון זמן קבוע של 7 ימים (Rigid Window) ===

=== מעקב סינון נתונים בגישה הקשיחה (Data Funnel) ===
1. התחלנו עם: 268 שאלוני PANAS תקינים בבסיס.
   -> נמחקו 53 שאלונים כי לא היו להם דיווחי EMA בכלל ב-7 הימים שקדמו להם.
   -> נמחקו 33 שאלונים בגלל חפיפת זמנים (פחות מ-7 ימים מהשאלון הקודם של הנבדק).
   -> נמחקו 17 שאלונים כי היו להם פחות מ-3 דיווחים (לא מספיק כדי לחשב ממוצע שבועי אמין).

✅ נשארנו עם: 165 שאלוני PANAS מוזהבים למודל בגישה הקשיחה.

✅ אגרגציה הושלמה: נוצרה טבלת rigid_wide_df עם 165 תצפיות ייחודיות.



,panas_original_id,user_id,panas_date,positive_affect_score,negative_affect_score,total_emas_in_window,mood_ALERT,mood_HAPPY,mood_NEUTRAL,mood_RESTED/RELAXED,mood_SAD,mood_TENSE/ANXIOUS,mood_TIRED
0,panas_0,621e2e8e67b776a24055b564,2021-07-26,37,14,8,0.000000,0.125000,0.625000,0.125000,0.000000,0.000000,0.125000
1,panas_1,621e2e8e67b776a24055b564,2021-05-31,38,12,14,0.000000,0.642857,0.000000,0.000000,0.000000,0.142857,0.214286
2,panas_100,621e30e467b776a240e817c7,2021-06-14,24,27,8,0.000000,0.000000,0.250000,0.000000,0.000000,0.000000,0.750000
3,panas_101,621e30e467b776a240e817c7,2021-07-05,33,22,3,0.000000,0.000000,0.000000,0.333333,0.000000,0.000000,0.666667
4,panas_102,621e310d67b776a24003096d,2021-12-27,33,19,6,0.000000,0.333333,0.000000,0.500000,0.000000,0.166667,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,panas_221,621e34db67b776a240c9c2be,2021-06-22,37,27,4,0.000000,0.000000,0.000000,0.000000,0.250000,0.500000,0.250000
96,panas_222,621e34ec67b776a240d60873,2021-06-02,24,25,9,0.000000,0.000000,0.000000,0.111111,0.111111,0.222222,0.555556
97,panas_227,621e351a67b776a240f6204b,2021-06-14,31,18,17,0.000000,0.117647,0.235294,0.294118,0.000000,0.000000,0.352941
98,panas_228,621e351a67b776a240f6204b,2021-07-26,37,10,18,0.000000,0.000000,0.000000,0.722222,0.000000,0.000000,0.277778


**# שלב 3: גישה ב' - חלון זמן דינאמי מותאם אישית (Dynamic Window) - הגישה המועדפת**
בגלל שנשארו לנו רק 165 תצפיות של שאלוני PANAS זה קצת נתונים. 
אנחנו רוצים לראות האם יש דרך להציל חלק מהנתונים.
חשבנו להציל את הנתונים של השבועות שיש בינהם חפיפה
אז עברנו לגישה שעושה קבוצות של מידע, כאשר חלון הזמן הוא דינמי. ככה שלפעמים חלון הזמן של הרגשות הוא קטן יותר מאשר 7.
 עד השאלון הקודם כדי למנוע זליגת נתונים.
# אנו מאפשרים מינימום של דיווח אחד, כי המודל יקבל את גודל החלון כפיצ'ר וידע להתמודד.
# (נצפה ל-214 שאלונים נקיים בסוף).

In [28]:
# =====================================================================
# שלב 3: גישה ב' - חלון זמן דינאמי מותאם אישית (Dynamic Window) - הגישה המועדפת
# =====================================================================
# בגישה זו אנו הולכים 7 ימים אחורה, *או* עד השאלון הקודם כדי למנוע זליגת נתונים.
# אנו מאפשרים מינימום של דיווח אחד, כי המודל יקבל את גודל החלון כפיצ'ר וידע להתמודד.
# (נצפה ל-214 שאלונים נקיים בסוף).
print("=== שלב 3: גישה ב' - חלון זמן דינאמי (Dynamic Window) ===")

dynamic_records = []

for _, p_row in panas_base.iterrows():
    # הגדרת חלון דינאמי למניעת Leakage
    if pd.notna(p_row['days_since_last_panas']) and p_row['days_since_last_panas'] < 7:
        days_to_look_back = p_row['days_since_last_panas']
    else:
        days_to_look_back = 7
        
    start_date = p_row['panas_date'] - pd.Timedelta(days=days_to_look_back)
    
    user_emas = ema_base[ema_base['user_id'] == p_row['user_id']]
    valid_emas = user_emas[(user_emas['ema_date'] > start_date) & (user_emas['ema_date'] <= p_row['panas_date'])]
    
    ema_count = len(valid_emas)
    
    # אין צורך לדאוג מחפיפה, שומרים כל שאלון שיש לו לפחות דיווח 1 בחלון הדינאמי שלו
    if ema_count >= 1:
        for _, ema_row in valid_emas.iterrows():
            dynamic_records.append({
                'user_id': p_row['user_id'],
                'panas_original_id': p_row['panas_original_id'],
                'panas_date': p_row['panas_date'],
                'positive_affect_score': p_row['positive_affect_score'],
                'negative_affect_score': p_row['negative_affect_score'],
                'window_days_size': days_to_look_back, # נשמר כפיצ'ר ללמידת המכונה
                'total_emas_in_window': ema_count,     # נשמר כפיצ'ר ללמידת המכונה
                'ema_original_id': ema_row['ema_original_id'],
                'data.MOOD': ema_row['data.MOOD']
            })

dynamic_long_df = pd.DataFrame(dynamic_records)

# ---------------------------------------------------------------------
# הרצת בדיקת שפיות ומעקב סינון נתונים (Data Funnel) עבור הגישה הדינאמית
# ---------------------------------------------------------------------
print("\n=== מעקב סינון נתונים בגישה הדינאמית (Data Funnel) ===")

dynamic_initial_count = len(panas_base)
print(f"1. התחלנו עם: {dynamic_initial_count} שאלוני PANAS תקינים בבסיס.")

# 1. כמה נמחקו כי לא היו להם דיווחים כלל בחלון הדינאמי?
dynamic_final_gold_count = dynamic_long_df['panas_original_id'].nunique()
dynamic_no_ema_count = dynamic_initial_count - dynamic_final_gold_count
print(f"   -> נמחקו {dynamic_no_ema_count} שאלונים כי לא היו להם דיווחי EMA כלל בחלון הזמן המותאם שלהם.")

# 2. דיווח על חפיפות (בגישה זו טיפלנו בהן אקטיבית)
print(f"   -> נמחקו 0 שאלונים בגלל חפיפת זמנים (החלון הדינאמי מונע זליגה באופן אוטומטי).")

# 3. דיווח על שאלונים דלילים (בגישה זו אנו שומרים אותם כפיצ'ר)
print(f"   -> נמחקו 0 שאלונים בגלל חוסר דיווחים (הורדנו את הרף לדיווח בודד, גודל החלון נשמר כפיצ'ר).")

print(f"\n✅ נשארנו עם: {dynamic_final_gold_count} שאלוני PANAS מוזהבים למודל בגישה הדינאמית.\n")

# ---------------------------------------------------------------------
# אגרגציה מפורמט ארוך לפורמט רחב (Wide Format) למודל
# ---------------------------------------------------------------------
mood_dummies_dyn = pd.get_dummies(dynamic_long_df['data.MOOD'], prefix='mood')
dynamic_wide_prep = pd.concat([dynamic_long_df, mood_dummies_dyn], axis=1)

agg_funcs_dyn = {
    'user_id': 'first', 'panas_date': 'first', 
    'positive_affect_score': 'first', 'negative_affect_score': 'first', 
    'window_days_size': 'first', 'total_emas_in_window': 'first'
}
for col in mood_dummies_dyn.columns:
    agg_funcs_dyn[col] = 'mean' # חישוב התפלגות הרגשות (אחוז מתוך הדיווחים)

dynamic_wide_df = dynamic_wide_prep.groupby('panas_original_id').agg(agg_funcs_dyn).reset_index()
print(f"✅ אגרגציה הושלמה: נוצרה טבלת dynamic_wide_df עם {len(dynamic_wide_df)} תצפיות ייחודיות (נטולות חפיפה).\n")
display(dynamic_wide_df.head(100))

=== שלב 3: גישה ב' - חלון זמן דינאמי (Dynamic Window) ===

=== מעקב סינון נתונים בגישה הדינאמית (Data Funnel) ===
1. התחלנו עם: 268 שאלוני PANAS תקינים בבסיס.
   -> נמחקו 54 שאלונים כי לא היו להם דיווחי EMA כלל בחלון הזמן המותאם שלהם.
   -> נמחקו 0 שאלונים בגלל חפיפת זמנים (החלון הדינאמי מונע זליגה באופן אוטומטי).
   -> נמחקו 0 שאלונים בגלל חוסר דיווחים (הורדנו את הרף לדיווח בודד, גודל החלון נשמר כפיצ'ר).

✅ נשארנו עם: 214 שאלוני PANAS מוזהבים למודל בגישה הדינאמית.

✅ אגרגציה הושלמה: נוצרה טבלת dynamic_wide_df עם 214 תצפיות ייחודיות (נטולות חפיפה).



,panas_original_id,user_id,panas_date,positive_affect_score,negative_affect_score,window_days_size,total_emas_in_window,mood_ALERT,mood_HAPPY,mood_NEUTRAL,mood_RESTED/RELAXED,mood_SAD,mood_TENSE/ANXIOUS,mood_TIRED
0,panas_0,621e2e8e67b776a24055b564,2021-07-26,37,14,7.0,8,0.000000,0.125000,0.625000,0.125000,0.0,0.000000,0.125000
1,panas_1,621e2e8e67b776a24055b564,2021-05-31,38,12,7.0,14,0.000000,0.642857,0.000000,0.000000,0.0,0.142857,0.214286
2,panas_10,621e2eaf67b776a2406b14ac,2022-01-07,26,16,2.0,6,0.166667,0.166667,0.000000,0.166667,0.0,0.333333,0.166667
3,panas_100,621e30e467b776a240e817c7,2021-06-14,24,27,7.0,8,0.000000,0.000000,0.250000,0.000000,0.0,0.000000,0.750000
4,panas_101,621e30e467b776a240e817c7,2021-07-05,33,22,7.0,3,0.000000,0.000000,0.000000,0.333333,0.0,0.000000,0.666667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,panas_192,621e33b067b776a240f39e56,2021-06-21,42,19,7.0,21,0.190476,0.285714,0.000000,0.285714,0.0,0.047619,0.190476
96,panas_193,621e33b067b776a240f39e56,2021-06-28,42,27,7.0,20,0.050000,0.300000,0.050000,0.100000,0.0,0.350000,0.150000
97,panas_194,621e33b067b776a240f39e56,2021-07-26,35,14,7.0,17,0.058824,0.705882,0.058824,0.000000,0.0,0.000000,0.176471
98,panas_195,621e33cf67b776a240087de9,2021-11-24,32,12,7.0,20,0.250000,0.000000,0.050000,0.250000,0.0,0.200000,0.250000


** שלב 4 (הנדסת מאפיינים ומניעת קוליניאריות).**

אורתוגונליות מושלמת: הורדנו עמודה אחת (mood_NEUTRAL), כך שעכשיו סכום השורות לא שווה בדיוק 1, מה שמאפשר לאלגוריתמים כמו Ridge או Linear Regression לחשב משקולות מדויקות מבלי לקרוס. בגלי המולטיקולנריות

אנטרופיה חכמה: המודל ילמד לזהות אנשים "קופצניים" רגשית לעומת אנשים "יציבים" כפיצ'ר נפרד, גם אם הממוצע של הרגשות שלהם זהה.
זה עוד מידע שאנחנו מוסיפים שאין לו תלות עם היחס בין כמויות המשתנים.


In [31]:
from scipy.stats import entropy
import pandas as pd
import numpy as np

print("=== שלב 4: הנדסת מאפיינים ומניעת קוליניאריות לשתי הגישות ===")

# =====================================================================
# 1. חישוב אנטרופיה (Emodiversity / תנודתיות רגשית)
# =====================================================================
# איתור עמודות הרגשות (אלו שמתחילות ב-'mood_')
mood_cols_rigid = [col for col in rigid_wide_df.columns if col.startswith('mood_')]
mood_cols_dynamic = [col for col in dynamic_wide_df.columns if col.startswith('mood_')]

# חישוב אנטרופיה לכל שורה (מוסיפים ערך אפסי 1e-9 כדי למנוע שגיאת log(0) ברגשות שקיבלו 0%)
rigid_wide_df['mood_volatility_entropy'] = rigid_wide_df[mood_cols_rigid].apply(lambda row: entropy(row + 1e-9), axis=1)
dynamic_wide_df['mood_volatility_entropy'] = dynamic_wide_df[mood_cols_dynamic].apply(lambda row: entropy(row + 1e-9), axis=1)

print("✅ מדד תנודתיות רגשית ('mood_volatility_entropy') חושב והתווסף לשתי הטבלאות.")

# =====================================================================
# 2. שמירה על אורתוגונליות (Drop Reference Category)
# =====================================================================
# אנו מסירים רגש אחד שישמש כ-Baseline (קבוצת ייחוס) כדי שהסכום לא יהיה בדיוק 1.
ref_mood = 'mood_NEUTRAL'

if ref_mood in rigid_wide_df.columns:
    rigid_wide_df.drop(columns=[ref_mood], inplace=True)
    print(f"✅ הרגש '{ref_mood}' הוסר מטבלת הגישה הקשיחה (RIGID).")

if ref_mood in dynamic_wide_df.columns:
    dynamic_wide_df.drop(columns=[ref_mood], inplace=True)
    print(f"✅ הרגש '{ref_mood}' הוסר מטבלת הגישה הדינמית (DYNAMIC).")

# עדכון רשימת הרגשות שנותרו למודל
final_mood_cols = [col for col in dynamic_wide_df.columns if col.startswith('mood_')]
print(f"\n📌 עמודות הרגש שנותרו לאימון המודל: \n{final_mood_cols}")

# =====================================================================
# הצגת התוצאות
# =====================================================================
print("\n--- הצצה לטבלה הדינמית לאחר הנדסת המאפיינים (מתמקדים בפיצ'רים החדשים) ---")
cols_to_show = ['panas_original_id', 'total_emas_in_window', 'mood_volatility_entropy'] + final_mood_cols
display(dynamic_wide_df[cols_to_show].head())

print("\n--- הצצה לטבלה הקשיחה לאחר הנדסת המאפיינים ---")
cols_to_show_rigid = ['panas_original_id', 'total_emas_in_window', 'mood_volatility_entropy'] + final_mood_cols
display(rigid_wide_df[cols_to_show_rigid].head())

=== שלב 4: הנדסת מאפיינים ומניעת קוליניאריות לשתי הגישות ===
✅ מדד תנודתיות רגשית ('mood_volatility_entropy') חושב והתווסף לשתי הטבלאות.

📌 עמודות הרגש שנותרו לאימון המודל: 
['mood_ALERT', 'mood_HAPPY', 'mood_RESTED/RELAXED', 'mood_SAD', 'mood_TENSE/ANXIOUS', 'mood_TIRED', 'mood_volatility_entropy']

--- הצצה לטבלה הדינמית לאחר הנדסת המאפיינים (מתמקדים בפיצ'רים החדשים) ---


,panas_original_id,total_emas_in_window,mood_volatility_entropy,mood_ALERT,mood_HAPPY,mood_RESTED/RELAXED,mood_SAD,mood_TENSE/ANXIOUS,mood_TIRED,mood_volatility_entropy
0,panas_0,8,0.856293,0.000000,0.125000,0.125000,0.0,0.000000,0.125000,0.856293
1,panas_1,14,1.163013,0.000000,0.642857,0.000000,0.0,0.142857,0.214286,1.163013
2,panas_10,6,1.278462,0.166667,0.166667,0.166667,0.0,0.333333,0.166667,1.278462
3,panas_100,8,0.682888,0.000000,0.000000,0.000000,0.0,0.000000,0.750000,0.682888
4,panas_101,3,1.057219,0.000000,0.000000,0.333333,0.0,0.000000,0.666667,1.057219



--- הצצה לטבלה הקשיחה לאחר הנדסת המאפיינים ---


,panas_original_id,total_emas_in_window,mood_volatility_entropy,mood_ALERT,mood_HAPPY,mood_RESTED/RELAXED,mood_SAD,mood_TENSE/ANXIOUS,mood_TIRED,mood_volatility_entropy
0,panas_0,8,0.856293,0.0,0.125000,0.125000,0.0,0.000000,0.125000,0.856293
1,panas_1,14,1.163013,0.0,0.642857,0.000000,0.0,0.142857,0.214286,1.163013
2,panas_100,8,0.682888,0.0,0.000000,0.000000,0.0,0.000000,0.750000,0.682888
3,panas_101,3,1.057219,0.0,0.000000,0.333333,0.0,0.000000,0.666667,1.057219
4,panas_102,6,1.195966,0.0,0.333333,0.500000,0.0,0.166667,0.000000,1.195966


** שלב 5 פיצול לקבוצות אימון ובדיקה**

In [32]:
import pandas as pd
import numpy as np
from scipy.stats import entropy
from sklearn.model_selection import train_test_split

print("=== שלב 5: פיצול מרובד ===")


# =====================================================================
# 1. פונקציית הפיצול המרובד לפי משתמשים (Stratified Group Split)
# =====================================================================
def stratified_group_split(df, user_col='user_id', test_size=0.2, random_state=42):
    """
    מפצלת Dataframe ל-Train ו-Test ברמת המשתמש (למניעת Data Leakage),
    תוך שמירה על יחס ייצוג שווה של משתמשים פעילים ופחות פעילים (Stratification).
    """
    # חישוב כמות השאלונים לכל משתמש
    user_stats = df.groupby(user_col).size().reset_index(name='panas_count')
    
    # חלוקה לדרגות פעילות (Low, Medium, High)
    user_stats['activity_level'] = pd.qcut(
        user_stats['panas_count'], 
        q=3, 
        labels=['Low', 'Medium', 'High'], 
        duplicates='drop'
    )
    
    # פיצול מרובד של המשתמשים
    train_users, test_users = train_test_split(
        user_stats[user_col], 
        test_size=test_size, 
        random_state=random_state, 
        stratify=user_stats['activity_level']
    )
    
    # חיתוך ה-DataFrame המקורי לפי קבוצות המשתמשים
    train_df = df[df[user_col].isin(train_users)].copy().reset_index(drop=True)
    test_df = df[df[user_col].isin(test_users)].copy().reset_index(drop=True)
    
    return train_df, test_df, user_stats

# =====================================================================
# 2. ביצוע הפיצול עבור הגישה הקשיחה (RIGID)
# =====================================================================
rigid_train_df, rigid_test_df, rigid_user_stats = stratified_group_split(
    rigid_wide_df, user_col='user_id', test_size=0.2, random_state=42
)

# =====================================================================
# 3. ביצוע הפיצול עבור הגישה הדינמית (DYNAMIC)
# =====================================================================
dynamic_train_df, dynamic_test_df, dynamic_user_stats = stratified_group_split(
    dynamic_wide_df, user_col='user_id', test_size=0.2, random_state=42
)

# =====================================================================
# 4. בדיקת שפיות (Sanity Checks) מקיפה לשתי הגישות
# =====================================================================
def run_sanity_check(train_df, test_df, approach_name):
    print(f"\n--- 🔍 בדיקת שפיות: גישת {approach_name} ---")
    
    train_users = set(train_df['user_id'])
    test_users = set(test_df['user_id'])
    overlap = train_users.intersection(test_users)
    
    # 1. וידוא חוסר חפיפה בין משתמשים
    if len(overlap) == 0:
        print(f"✅ תקין: 0 משתמשים חופפים בין Train ל-Test (אין Data Leakage).")
    else:
        print(f"❌ שגיאה: נמצאו {len(overlap)} משתמשים חופפים!")
        
    # 2. ספירת תצפיות ומשתמשים
    print(f"📊 סה\"כ תצפיות ב-Train: {len(train_df)} ({len(train_users)} משתמשים)")
    print(f"📊 סה\"כ תצפיות ב-Test:  {len(test_df)} ({len(test_users)} משתמשים)")
    
    # 3. יחס תצפיות ב-Test מתוך הסך הכל
    total_obs = len(train_df) + len(test_df)
    test_ratio = len(test_df) / total_obs if total_obs > 0 else 0
    print(f"📈 אחוז תצפיות ב-Test: {test_ratio:.1%}")
    
    # 4. ממוצע שאלונים למשתמש בכל קבוצה (אינדיקציה לריבוד מוצלח)
    mean_train = train_df.groupby('user_id').size().mean()
    mean_test = test_df.groupby('user_id').size().mean()
    print(f"⚖️  ממוצע שאלוני PANAS למשתמש: Train = {mean_train:.2f} | Test = {mean_test:.2f}")

# הרצת בדיקות השפיות
run_sanity_check(rigid_train_df, rigid_test_df, "קבועה (RIGID)")
run_sanity_check(dynamic_train_df, dynamic_test_df, "דינמית (DYNAMIC)")

# =====================================================================
# 5. הצגת דגמי הנתונים בסוף (Preview of Final Datasets)
# =====================================================================
print("\n" + "="*60)
print("📌 הצגת נתונים בסוף (Data Preview)")
print("="*60)

print("\n--- 1. RIGID Train Dataset (rigid_train_df) ---")
display(rigid_train_df.head(3))

print("\n--- 2. RIGID Test Dataset (rigid_test_df) ---")
display(rigid_test_df.head(3))

print("\n--- 3. DYNAMIC Train Dataset (dynamic_train_df) ---")
display(dynamic_train_df.head(3))

print("\n--- 4. DYNAMIC Test Dataset (dynamic_test_df) ---")
display(dynamic_test_df.head(3))

=== שלב 5: פיצול מרובד ===

--- 🔍 בדיקת שפיות: גישת קבועה (RIGID) ---
✅ תקין: 0 משתמשים חופפים בין Train ל-Test (אין Data Leakage).
📊 סה"כ תצפיות ב-Train: 132 (32 משתמשים)
📊 סה"כ תצפיות ב-Test:  33 (9 משתמשים)
📈 אחוז תצפיות ב-Test: 20.0%
⚖️  ממוצע שאלוני PANAS למשתמש: Train = 4.12 | Test = 3.67

--- 🔍 בדיקת שפיות: גישת דינמית (DYNAMIC) ---
✅ תקין: 0 משתמשים חופפים בין Train ל-Test (אין Data Leakage).
📊 סה"כ תצפיות ב-Train: 166 (36 משתמשים)
📊 סה"כ תצפיות ב-Test:  48 (10 משתמשים)
📈 אחוז תצפיות ב-Test: 22.4%
⚖️  ממוצע שאלוני PANAS למשתמש: Train = 4.61 | Test = 4.80

📌 הצגת נתונים בסוף (Data Preview)

--- 1. RIGID Train Dataset (rigid_train_df) ---


,panas_original_id,user_id,panas_date,positive_affect_score,negative_affect_score,total_emas_in_window,mood_ALERT,mood_HAPPY,mood_RESTED/RELAXED,mood_SAD,mood_TENSE/ANXIOUS,mood_TIRED,mood_volatility_entropy
0,panas_0,621e2e8e67b776a24055b564,2021-07-26,37,14,8,0.0,0.125000,0.125,0.0,0.000000,0.125000,0.856293
1,panas_1,621e2e8e67b776a24055b564,2021-05-31,38,12,14,0.0,0.642857,0.000,0.0,0.142857,0.214286,1.163013
2,panas_100,621e30e467b776a240e817c7,2021-06-14,24,27,8,0.0,0.000000,0.000,0.0,0.000000,0.750000,0.682888



--- 2. RIGID Test Dataset (rigid_test_df) ---


,panas_original_id,user_id,panas_date,positive_affect_score,negative_affect_score,total_emas_in_window,mood_ALERT,mood_HAPPY,mood_RESTED/RELAXED,mood_SAD,mood_TENSE/ANXIOUS,mood_TIRED,mood_volatility_entropy
0,panas_119,621e324e67b776a2400191cb,2021-11-27,43,13,19,0.263158,0.052632,0.210526,0.000000,0.105263,0.157895,1.099779
1,panas_120,621e324e67b776a2400191cb,2021-12-20,46,13,19,0.421053,0.105263,0.210526,0.052632,0.052632,0.000000,1.111698
2,panas_121,621e324e67b776a2400191cb,2022-01-09,49,12,14,0.285714,0.071429,0.428571,0.000000,0.000000,0.000000,1.023345



--- 3. DYNAMIC Train Dataset (dynamic_train_df) ---


,panas_original_id,user_id,panas_date,positive_affect_score,negative_affect_score,window_days_size,total_emas_in_window,mood_ALERT,mood_HAPPY,mood_RESTED/RELAXED,mood_SAD,mood_TENSE/ANXIOUS,mood_TIRED,mood_volatility_entropy
0,panas_10,621e2eaf67b776a2406b14ac,2022-01-07,26,16,2.0,6,0.166667,0.166667,0.166667,0.0,0.333333,0.166667,1.278462
1,panas_100,621e30e467b776a240e817c7,2021-06-14,24,27,7.0,8,0.000000,0.000000,0.000000,0.0,0.000000,0.750000,0.682888
2,panas_101,621e30e467b776a240e817c7,2021-07-05,33,22,7.0,3,0.000000,0.000000,0.333333,0.0,0.000000,0.666667,1.057219



--- 4. DYNAMIC Test Dataset (dynamic_test_df) ---


,panas_original_id,user_id,panas_date,positive_affect_score,negative_affect_score,window_days_size,total_emas_in_window,mood_ALERT,mood_HAPPY,mood_RESTED/RELAXED,mood_SAD,mood_TENSE/ANXIOUS,mood_TIRED,mood_volatility_entropy
0,panas_0,621e2e8e67b776a24055b564,2021-07-26,37,14,7.0,8,0.00,0.125000,0.125,0.00,0.000000,0.125000,0.856293
1,panas_1,621e2e8e67b776a24055b564,2021-05-31,38,12,7.0,14,0.00,0.642857,0.000,0.00,0.142857,0.214286,1.163013
2,panas_113,621e323667b776a240f19134,2021-11-22,41,23,7.0,20,0.05,0.150000,0.350,0.05,0.050000,0.350000,1.269953
